# LeetCode 941 — Valid Mountain Array bằng Python

**Link đề:** https://leetcode.com/problems/valid-mountain-array/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def validMountainArray(self, arr: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Đọc chính xác định nghĩa "mảng núi": tăng **nghiêm ngặt** rồi giảm **nghiêm ngặt**, đỉnh không ở hai đầu.
2. Nền tảng: so sánh cặp phần tử kề nhau, `zip(arr, arr[1:])`, `all`, vòng `while` có kiểm tra biên.
3. Đi từ cách "tìm đỉnh bằng `max` rồi kiểm tra hai phía" → **leo núi một lượt** (walk up / walk down) → **hai người leo từ hai đầu** → máy trạng thái.
4. Trace, chứng minh tính đúng, Time/Space, lỗi kinh điển (đỉnh ở đầu/cuối, đoạn bằng nhau).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — mảng có phải hình "ngọn núi"?

`arr` là **mảng núi** khi và chỉ khi:
- `len(arr) >= 3`, và
- tồn tại chỉ số `i` với `0 < i < len(arr) - 1` sao cho
  - `arr[0] < arr[1] < ... < arr[i]` (đi lên **nghiêm ngặt**), và
  - `arr[i] > arr[i+1] > ... > arr[len-1]` (đi xuống **nghiêm ngặt**).

| `arr` | Output | Lý do |
|---|:---:|---|
| `[2,1]` | `False` | ít hơn 3 phần tử |
| `[3,5,5]` | `False` | `5, 5` bằng nhau — không nghiêm ngặt |
| `[0,3,2,1]` | `True` | lên 0→3, xuống 3→2→1 |
| `[0,1,2,3]` | `False` | chỉ đi lên, đỉnh ở cuối |
| `[3,2,1]` | `False` | chỉ đi xuống, đỉnh ở đầu |

**Constraints:** `1 <= arr.length <= 10^4`, `0 <= arr[i] <= 10^4`.

In [1]:
# "Vẽ" ngọn núi bằng ký tự để dễ hình dung
def draw(arr):
    top = max(arr)
    for level in range(top, 0, -1):
        print("".join(" # " if v >= level else "   " for v in arr))
    print("".join(f"{v:^3}" for v in arr))

draw([0, 3, 2, 1])
print()
draw([3, 5, 5])

    #       
    #  #    
    #  #  # 
 0  3  2  1 

    #  # 
    #  # 
 #  #  # 
 #  #  # 
 #  #  # 
 3  5  5 


`f"{v:^3}"` căn **giữa** `v` trong 3 ký tự. `range(top, 0, -1)` đếm ngược `top, top-1, ..., 1`.

## 2. Nền tảng Python cần dùng

### 2.1. So sánh các cặp kề nhau bằng `zip(arr, arr[1:])`

`arr[1:]` là mảng bỏ phần tử đầu. `zip(arr, arr[1:])` cho các cặp `(arr[0], arr[1]), (arr[1], arr[2]), ...` — đúng các cặp **kề nhau**.

In [2]:
arr = [0, 3, 2, 1]
pairs = list(zip(arr, arr[1:]))
print(pairs)
for a, b_ in pairs:
    print(a, b_, "lên" if a < b_ else ("xuống" if a > b_ else "bằng"))

[(0, 3), (3, 2), (2, 1)]
0 3 lên
3 2 xuống
2 1 xuống


### 2.2. `all` để kiểm tra "tăng nghiêm ngặt"

`all(a < b for a, b in zip(seg, seg[1:]))` → `True` nếu đoạn `seg` tăng nghiêm ngặt. Đoạn 0 hoặc 1 phần tử cho `True` (không có cặp nào vi phạm).

In [3]:
def strictly_increasing(seg):
    return all(a < b for a, b in zip(seg, seg[1:]))

def strictly_decreasing(seg):
    return all(a > b for a, b in zip(seg, seg[1:]))

print(strictly_increasing([0, 3]), strictly_increasing([3, 5, 5]), strictly_increasing([7]))
print(strictly_decreasing([3, 2, 1]), strictly_decreasing([5, 5]))

True False True
True False


### 2.3. `max` và `list.index` — tìm vị trí đỉnh

`max(arr)` cho giá trị lớn nhất; `arr.index(x)` cho vị trí **đầu tiên** của `x`. Chú ý: nếu đỉnh xuất hiện 2 lần (như `[1,5,5,1]`), `index` chỉ trả lần đầu.

In [4]:
arr = [0, 3, 2, 1]
peak = arr.index(max(arr))
print("max =", max(arr), "| vị trí đỉnh =", peak)
print([1, 5, 5, 1].index(5), "<- chỉ lần xuất hiện đầu")

max = 3 | vị trí đỉnh = 1
1 <- chỉ lần xuất hiện đầu


### 2.4. Vòng `while` có kiểm tra biên

Mẫu "đi tiếp chừng nào còn thỏa": `while i + 1 < n and arr[i] < arr[i + 1]: i += 1`.
Điều kiện biên `i + 1 < n` phải đứng **trước** để tránh đọc `arr[n]` (nhờ đánh giá ngắn mạch của `and`).

In [5]:
arr = [1, 2, 3]
i, n = 0, len(arr)
while i + 1 < n and arr[i] < arr[i + 1]:
    i += 1
print("dừng ở i =", i, "(không lỗi dù đi tới cuối)")

dừng ở i = 2 (không lỗi dù đi tới cuối)


## 3. Cách trực giác — tìm đỉnh rồi kiểm tra hai sườn

1. Nếu `len(arr) < 3` → `False`.
2. Đỉnh là vị trí của giá trị lớn nhất: `p = arr.index(max(arr))`.
3. Đỉnh không được ở hai đầu: `0 < p < n - 1`.
4. Sườn trái `arr[:p+1]` tăng nghiêm ngặt, sườn phải `arr[p:]` giảm nghiêm ngặt.

Vì sao đỉnh **phải** là giá trị lớn nhất? Trong mảng núi, mọi phần tử bên trái nhỏ hơn đỉnh (đi lên), mọi phần tử bên phải nhỏ hơn đỉnh (đi xuống).

In [6]:
def mountain_by_max(arr: list[int]) -> bool:
    n = len(arr)
    if n < 3:
        return False
    p = arr.index(max(arr))
    if p == 0 or p == n - 1:
        return False
    return strictly_increasing(arr[:p + 1]) and strictly_decreasing(arr[p:])

for arr in [[2, 1], [3, 5, 5], [0, 3, 2, 1], [0, 1, 2, 3], [3, 2, 1], [1, 5, 5, 1]]:
    print(arr, "->", mountain_by_max(arr))

[2, 1] -> False
[3, 5, 5] -> False
[0, 3, 2, 1] -> True
[0, 1, 2, 3] -> False
[3, 2, 1] -> False
[1, 5, 5, 1] -> False


### 3.1. Chi phí

`max` duyệt 1 lần, `index` duyệt tới đỉnh, hai lát cắt `arr[:p+1]`, `arr[p:]` tạo bản sao (`O(n)` bộ nhớ), hai lần `all` duyệt thêm. Vẫn `O(n)` thời gian nhưng **duyệt nhiều lượt** và **tốn bộ nhớ** cho slice.

Câu hỏi: có thể làm trong **một lượt**, `O(1)` bộ nhớ?

## 4. Nút thắt và insight — "leo núi" một lượt

**Hình dung:** một người leo từ chân trái.
1. **Leo lên** chừng nào bước tiếp theo **cao hơn**: `while arr[i] < arr[i+1]: i += 1`. Dừng ở đỉnh `i`.
2. Đỉnh **không được** ở chân trái (`i == 0` nghĩa là không leo được bước nào) hoặc chân phải (`i == n-1` nghĩa là chỉ đi lên).
3. **Đi xuống** chừng nào bước tiếp theo **thấp hơn**: `while arr[i] > arr[i+1]: i += 1`.
4. Mảng núi ⇔ người leo **tới được chân phải** (`i == n - 1`).

Gặp "bằng nhau" thì cả hai vòng đều dừng → không tới được cuối → `False`. Không cần xử lý riêng!

## 5. Lời giải chính — walk up, walk down

**Mảnh 1:** `n = len(arr)`, `i = 0`.

**Mảnh 2 — leo lên:** `while i + 1 < n and arr[i] < arr[i + 1]: i += 1`.

**Mảnh 3 — đỉnh hợp lệ:** `if i == 0 or i == n - 1: return False`. (Điều này cũng loại luôn `n < 3`: với `n = 1` thì `i = 0`; với `n = 2` thì `i` là 0 hoặc 1 = `n - 1`.)

**Mảnh 4 — xuống dốc:** `while i + 1 < n and arr[i] > arr[i + 1]: i += 1`.

**Mảnh 5:** `return i == n - 1`.

In [7]:
def mountain_walk(arr: list[int]) -> bool:
    n = len(arr)
    i = 0
    while i + 1 < n and arr[i] < arr[i + 1]:     # leo lên
        i += 1
    if i == 0 or i == n - 1:                       # đỉnh ở chân núi
        return False
    while i + 1 < n and arr[i] > arr[i + 1]:     # xuống dốc
        i += 1
    return i == n - 1

for arr in [[2, 1], [3, 5, 5], [0, 3, 2, 1], [0, 1, 2, 3], [3, 2, 1], [1], [1, 3, 2]]:
    print(arr, "->", mountain_walk(arr))

[2, 1] -> False
[3, 5, 5] -> False
[0, 3, 2, 1] -> True
[0, 1, 2, 3] -> False
[3, 2, 1] -> False
[1] -> False
[1, 3, 2] -> True


## 6. TRACE từng bước

Ví dụ `arr = [0, 2, 3, 4, 5, 2, 1, 0]` (n = 8):

| Giai đoạn | i | arr[i] | arr[i+1] | So sánh | Hành động |
|---|:---:|:---:|:---:|---|---|
| lên | 0 | 0 | 2 | < | i=1 |
| lên | 1 | 2 | 3 | < | i=2 |
| lên | 2 | 3 | 4 | < | i=3 |
| lên | 3 | 4 | 5 | < | i=4 |
| lên | 4 | 5 | 2 | không < | dừng, đỉnh i=4 (0 < 4 < 7 ✓) |
| xuống | 4 | 5 | 2 | > | i=5 |
| xuống | 5 | 2 | 1 | > | i=6 |
| xuống | 6 | 1 | 0 | > | i=7 |
| xuống | 7 | — | — | hết mảng | dừng, i = 7 = n-1 → **True** |

Ví dụ `[0, 2, 3, 3, 5, 2, 1, 0]`: leo lên dừng ở `i=2` (3 không < 3), xuống dốc cũng dừng ngay (3 không > 3) → `i = 2 ≠ 7` → **False**.

In [8]:
def trace_mountain(arr):
    n = len(arr)
    i = 0
    print("arr =", arr)
    while i + 1 < n and arr[i] < arr[i + 1]:
        print(f"  lên  : arr[{i}]={arr[i]} < arr[{i+1}]={arr[i+1]} -> i={i+1}")
        i += 1
    print(f"  đỉnh tại i={i}")
    if i == 0 or i == n - 1:
        print("  đỉnh ở chân núi -> False")
        return False
    while i + 1 < n and arr[i] > arr[i + 1]:
        print(f"  xuống: arr[{i}]={arr[i]} > arr[{i+1}]={arr[i+1]} -> i={i+1}")
        i += 1
    print(f"  dừng ở i={i}, n-1={n-1} -> {i == n - 1}")
    return i == n - 1

trace_mountain([0, 2, 3, 4, 5, 2, 1, 0])
print()
trace_mountain([0, 2, 3, 3, 5, 2, 1, 0])
print()
trace_mountain([0, 1, 2, 3])

arr = [0, 2, 3, 4, 5, 2, 1, 0]
  lên  : arr[0]=0 < arr[1]=2 -> i=1
  lên  : arr[1]=2 < arr[2]=3 -> i=2
  lên  : arr[2]=3 < arr[3]=4 -> i=3
  lên  : arr[3]=4 < arr[4]=5 -> i=4
  đỉnh tại i=4
  xuống: arr[4]=5 > arr[5]=2 -> i=5
  xuống: arr[5]=2 > arr[6]=1 -> i=6
  xuống: arr[6]=1 > arr[7]=0 -> i=7
  dừng ở i=7, n-1=7 -> True

arr = [0, 2, 3, 3, 5, 2, 1, 0]
  lên  : arr[0]=0 < arr[1]=2 -> i=1
  lên  : arr[1]=2 < arr[2]=3 -> i=2
  đỉnh tại i=2
  dừng ở i=2, n-1=7 -> False

arr = [0, 1, 2, 3]
  lên  : arr[0]=0 < arr[1]=1 -> i=1
  lên  : arr[1]=1 < arr[2]=2 -> i=2
  lên  : arr[2]=2 < arr[3]=3 -> i=3
  đỉnh tại i=3
  đỉnh ở chân núi -> False


False

In [9]:
# Trường hợp "núi có 2 đỉnh": xuống rồi lại lên
trace_mountain([1, 3, 2, 4, 1])

arr = [1, 3, 2, 4, 1]
  lên  : arr[0]=1 < arr[1]=3 -> i=1
  đỉnh tại i=1
  xuống: arr[1]=3 > arr[2]=2 -> i=2
  dừng ở i=2, n-1=4 -> False


False

## 7. Các lời giải khác và bảng so sánh

### 7.1. Hai người leo từ hai phía

Người A leo từ trái lên (`i` tăng khi `arr[i] < arr[i+1]`), người B leo từ phải lên (`j` giảm khi `arr[j-1] > arr[j]`). Mảng núi ⇔ hai người **gặp nhau tại cùng một đỉnh**, và đỉnh không ở hai đầu.

In [10]:
def mountain_two_climbers(arr: list[int]) -> bool:
    n = len(arr)
    i, j = 0, n - 1
    while i + 1 < n and arr[i] < arr[i + 1]:
        i += 1
    while j > 0 and arr[j - 1] > arr[j]:
        j -= 1
    return 0 < i == j < n - 1

for arr in [[2, 1], [3, 5, 5], [0, 3, 2, 1], [0, 1, 2, 3], [1, 3, 2, 4, 1]]:
    print(arr, "->", mountain_two_climbers(arr))

[2, 1] -> False
[3, 5, 5] -> False
[0, 3, 2, 1] -> True
[0, 1, 2, 3] -> False
[1, 3, 2, 4, 1] -> False


`0 < i == j < n - 1` là so sánh chuỗi: tương đương `0 < i and i == j and j < n - 1`.

### 7.2. Máy trạng thái (state machine) duyệt các cặp kề nhau

Trạng thái: `"up"` (đang lên) → `"down"` (đang xuống). Quy tắc:
- Cặp bằng nhau → `False`.
- Đang `up` gặp cặp giảm: nếu chưa từng lên → `False` (đỉnh ở đầu); ngược lại chuyển sang `down`.
- Đang `down` gặp cặp tăng → `False` (lên lại).
- Kết thúc: phải ở trạng thái `down`.

In [11]:
def mountain_state_machine(arr: list[int]) -> bool:
    if len(arr) < 3:
        return False
    state = "up"
    went_up = False
    for a, b_ in zip(arr, arr[1:]):
        if a == b_:
            return False
        if state == "up":
            if a < b_:
                went_up = True
            elif not went_up:
                return False
            else:
                state = "down"
        else:                       # state == "down"
            if a < b_:
                return False
    return state == "down"

for arr in [[2, 1], [3, 5, 5], [0, 3, 2, 1], [0, 1, 2, 3], [3, 2, 1], [1, 3, 2, 4, 1]]:
    print(arr, "->", mountain_state_machine(arr))

[2, 1] -> False
[3, 5, 5] -> False
[0, 3, 2, 1] -> True
[0, 1, 2, 3] -> False
[3, 2, 1] -> False
[1, 3, 2, 4, 1] -> False


### 7.3. Đếm "đổi hướng" trên dãy dấu chênh lệch

Biến mỗi cặp kề thành dấu: `+1` (lên), `-1` (xuống), `0` (bằng). Mảng núi ⇔ không có `0`, dãy dấu có dạng `+ + ... + - - ... -` với ít nhất một `+` và một `-`. Tức là: dấu đầu là `+`, dấu cuối là `-`, và chỉ **đổi dấu đúng một lần**.

In [12]:
def mountain_signs(arr: list[int]) -> bool:
    signs = [(b_ > a) - (b_ < a) for a, b_ in zip(arr, arr[1:])]
    if not signs or 0 in signs:
        return False
    changes = sum(1 for s, t in zip(signs, signs[1:]) if s != t)
    return signs[0] == 1 and signs[-1] == -1 and changes == 1

print([(b_ > a) - (b_ < a) for a, b_ in zip([0, 3, 2, 1], [3, 2, 1])])
for arr in [[2, 1], [3, 5, 5], [0, 3, 2, 1], [0, 1, 2, 3], [1, 3, 2, 4, 1]]:
    print(arr, "->", mountain_signs(arr))

[1, -1, -1]
[2, 1] -> False
[3, 5, 5] -> False
[0, 3, 2, 1] -> True
[0, 1, 2, 3] -> False
[1, 3, 2, 4, 1] -> False


`(b > a) - (b < a)` là mẹo tính **dấu** trong Python: `True - False = 1`, `False - True = -1`, `False - False = 0`.

### 7.4. Bảng so sánh

`n = len(arr)`.

| Cách | Time | Space phụ | Số lượt duyệt | Ghi chú |
|---|---|---|:---:|---|
| Tìm đỉnh bằng max + slice | `O(n)` | `O(n)` | ~4 | dễ nghĩ |
| **Walk up / walk down** | `O(n)` | `O(1)` | 1 | nên nộp |
| Hai người leo | `O(n)` | `O(1)` | ≤ 2 | đẹp, dễ chứng minh |
| Máy trạng thái | `O(n)` | `O(1)` | 1 | tổng quát hóa tốt |
| Dãy dấu | `O(n)` | `O(n)` | 2 | trực quan |

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def validMountainArray(self, arr: list[int]) -> bool:
        n = len(arr)
        i = 0
        while i + 1 < n and arr[i] < arr[i + 1]:
            i += 1
        if i == 0 or i == n - 1:
            return False
        while i + 1 < n and arr[i] > arr[i + 1]:
            i += 1
        return i == n - 1

sol = Solution()
assert sol.validMountainArray([2, 1]) == False
assert sol.validMountainArray([3, 5, 5]) == False
assert sol.validMountainArray([0, 3, 2, 1]) == True
print("Solution PASS 3 ví dụ đề bài")

Solution PASS 3 ví dụ đề bài


## 9. Vì sao thuật toán đúng?

Gọi `p` là vị trí dừng sau vòng leo lên.

**Sau vòng 1:** `arr[0] < arr[1] < ... < arr[p]` (mọi bước đã đi đều là bước lên), và hoặc `p = n - 1`, hoặc `arr[p] >= arr[p+1]`.

**(⇒) Nếu trả `True`:** `0 < p < n - 1` và vòng 2 đi tới `n - 1`, nên `arr[p] > arr[p+1] > ... > arr[n-1]`. Đó đúng là định nghĩa mảng núi với đỉnh `i = p`. Hơn nữa `n >= 3` vì `0 < p < n - 1`.

**(⇐) Nếu `arr` là mảng núi với đỉnh `i*`:** vòng 1 đi lên qua mọi bước tăng và dừng đúng tại `i*` (vì `arr[i*] > arr[i*+1]`), nên `p = i*` với `0 < p < n-1`. Vòng 2 đi qua toàn bộ sườn giảm tới `n - 1`. Trả `True`.

Đỉnh là **duy nhất**: vòng 1 luôn dừng ở **đỉnh đầu tiên**; nếu sau đó còn đoạn đi lên (núi 2 đỉnh) hoặc đoạn bằng, vòng 2 dừng sớm → `False`. ∎

## 10. Phân tích Time / Space

- **Time `O(n)`:** biến `i` chỉ tăng, tổng hai vòng `while` tối đa `n - 1` bước.
- **Space `O(1)`:** chỉ dùng `i`, `n`.

So với cách `max` + slice: cùng `O(n)` nhưng tạo hai mảng con `O(n)` và duyệt nhiều lượt.

In [14]:
import time, random
rng = random.Random(941)
n = 10**4
peak = n // 2
big = list(range(peak)) + list(range(peak, -1, -1))[: n - peak]
print("len =", len(big), "| là núi?", mountain_walk(big))
for f in [mountain_by_max, mountain_walk, mountain_two_climbers, mountain_state_machine, mountain_signs]:
    t0 = time.perf_counter()
    for _ in range(100):
        f(big)
    print(f"{f.__name__:24} {time.perf_counter() - t0:.4f}s / 100 lần")

len = 10000 | là núi? True
mountain_by_max          0.0674s / 100 lần


mountain_walk            0.0829s / 100 lần


mountain_two_climbers    0.0793s / 100 lần
mountain_state_machine   0.0534s / 100 lần


mountain_signs           0.0787s / 100 lần


## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. Độ dài 1, 2, 3.
3. Chỉ tăng / chỉ giảm (đỉnh ở đầu hoặc cuối).
4. Đoạn bằng ở sườn lên, ở đỉnh, ở sườn xuống.
5. Núi hai đỉnh.
6. Ngẫu nhiên có seed, đối chiếu với định nghĩa gốc (brute force thử mọi `i`).

In [15]:
def mountain_definition(arr):
    # Kiểm tra trực tiếp theo định nghĩa: thử mọi đỉnh i
    n = len(arr)
    if n < 3:
        return False
    for i in range(1, n - 1):
        if all(arr[k] < arr[k + 1] for k in range(i)) and all(arr[k] > arr[k + 1] for k in range(i, n - 1)):
            return True
    return False

tests = [
    ([2, 1], False), ([3, 5, 5], False), ([0, 3, 2, 1], True),
    ([1], False), ([1, 2], False), ([1, 3, 2], True), ([1, 2, 3], False), ([3, 2, 1], False),
    ([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], False), ([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], False),
    ([1, 1, 2, 1], False), ([1, 2, 2, 1], False), ([1, 2, 1, 1], False),
    ([1, 3, 2, 4, 1], False), ([0, 2, 3, 4, 5, 2, 1, 0], True), ([5, 5, 5], False),
]
solvers = [mountain_by_max, mountain_walk, mountain_two_climbers, mountain_state_machine,
           mountain_signs, mountain_definition, Solution().validMountainArray]
for arr, expected in tests:
    for f in solvers:
        got = f(arr)
        assert got == expected, (f.__name__, arr, got, expected)
print(f"PASS {len(tests)} test x {len(solvers)} cách")

PASS 16 test x 7 cách


In [16]:
cnt_true = 0
for _ in range(3000):
    arr = [rng.randint(0, 4) for _ in range(rng.randint(1, 7))]
    expected = mountain_definition(arr)
    cnt_true += expected
    for f in solvers:
        assert f(arr) == expected, (f.__name__, arr)
print(f"PASS 3000 mảng ngẫu nhiên ({cnt_true} mảng núi)")

PASS 3000 mảng ngẫu nhiên (184 mảng núi)


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên kiểm tra đỉnh ở hai đầu

In [17]:
def wrong_no_peak_check(arr):
    n, i = len(arr), 0
    while i + 1 < n and arr[i] < arr[i + 1]:
        i += 1
    while i + 1 < n and arr[i] > arr[i + 1]:
        i += 1
    return i == n - 1

for arr in [[0, 1, 2, 3], [3, 2, 1], [1]]:
    print(arr, "wrong ->", wrong_no_peak_check(arr), "| đúng ->", mountain_walk(arr))

[0, 1, 2, 3] wrong -> True | đúng -> False
[3, 2, 1] wrong -> True | đúng -> False
[1] wrong -> True | đúng -> False


**Vì sao sai:** mảng chỉ tăng hoặc chỉ giảm vẫn "đi tới cuối". **Sửa:** sau khi leo, kiểm tra `i == 0 or i == n - 1` → `False`.

### Lỗi 2: Dùng `<=` / `>=` (không nghiêm ngặt)

In [18]:
def wrong_non_strict(arr):
    n, i = len(arr), 0
    while i + 1 < n and arr[i] <= arr[i + 1]:
        i += 1
    if i == 0 or i == n - 1:
        return False
    while i + 1 < n and arr[i] >= arr[i + 1]:
        i += 1
    return i == n - 1

print("wrong_non_strict([1,2,2,1]) =", wrong_non_strict([1, 2, 2, 1]), "| đúng: False")

wrong_non_strict([1,2,2,1]) = True | đúng: False


### Lỗi 3: Thiếu kiểm tra biên `i + 1 < n` → `IndexError`

In [19]:
def wrong_no_bound(arr):
    i = 0
    while arr[i] < arr[i + 1]:
        i += 1
    return i

try:
    wrong_no_bound([1, 2, 3])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 4: Tìm đỉnh bằng `max` nhưng quên rằng đỉnh có thể lặp

`[1, 5, 5, 1]`: `index(max)` trả vị trí 1; nếu chỉ kiểm tra sườn trái tăng và **quên** kiểm tra sườn phải giảm nghiêm ngặt (`5, 5`), sẽ trả `True` sai. Cách walk tự động xử lý vì so sánh nghiêm ngặt.

### Lỗi 5: Chỉ đếm số lần "đổi hướng" mà quên chiều

Một lần đổi hướng nhưng là `xuống rồi lên` (thung lũng `[3, 1, 2]`) không phải núi. Phải kiểm tra **dấu đầu là lên**.

In [20]:
def wrong_only_count_changes(arr):
    signs = [(b_ > a) - (b_ < a) for a, b_ in zip(arr, arr[1:])]
    return bool(signs) and 0 not in signs and sum(1 for s, t in zip(signs, signs[1:]) if s != t) == 1

print("thung lũng [3,1,2]:", wrong_only_count_changes([3, 1, 2]), "| đúng:", mountain_walk([3, 1, 2]))

thung lũng [3,1,2]: True | đúng: False


## 13. Mẹo nhận dạng pattern

- Đề mô tả **hình dạng** của dãy (tăng rồi giảm, zigzag, đơn điệu) → duyệt **các cặp kề nhau** và theo dõi **pha / trạng thái**.
- "Tăng nghiêm ngặt" → `<`; "không giảm" → `<=`. Đọc kỹ từng chữ.
- Mẫu **walk while condition**: `while i + 1 < n and cond(arr[i], arr[i+1]): i += 1` — dùng lại cho mọi bài "đi hết một đoạn đơn điệu".
- Kiểm tra biên ngay sau mỗi pha (đỉnh không ở đầu/cuối).

Bài liên quan: 852 Peak Index in a Mountain Array (tìm nhị phân), 845 Longest Mountain in Array, 1671 Minimum Removals to Make Mountain Array, 896 Monotonic Array, 978 Longest Turbulent Subarray.

## 14. Biến thể

### 14.1. Monotonic Array (LeetCode 896) — dãy đơn điệu (không giảm hoặc không tăng)

In [21]:
def is_monotonic(arr):
    inc = all(a <= b_ for a, b_ in zip(arr, arr[1:]))
    dec = all(a >= b_ for a, b_ in zip(arr, arr[1:]))
    return inc or dec

print(is_monotonic([1, 2, 2, 3]), is_monotonic([6, 5, 4, 4]), is_monotonic([1, 3, 2]))

True True False


### 14.2. Peak Index in a Mountain Array (LeetCode 852) — biết chắc là núi, tìm đỉnh trong `O(log n)`

Tìm nhị phân: nếu `arr[mid] < arr[mid+1]` thì đỉnh ở **bên phải** `mid`; ngược lại đỉnh ở `mid` hoặc bên trái.

In [22]:
def peak_index(arr):
    lo, hi = 0, len(arr) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < arr[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

print(peak_index([0, 2, 3, 4, 5, 2, 1, 0]), peak_index([0, 10, 5, 2]))

4 1


### 14.3. Longest Mountain in Array (LeetCode 845) — độ dài mảng con núi dài nhất

Từ mỗi "chân", leo lên rồi xuống bằng chính kỹ thuật walk; nhảy `i` tới cuối núi vừa xét.

In [23]:
def longest_mountain(arr):
    n, best, base = len(arr), 0, 0
    while base < n:
        end = base
        if end + 1 < n and arr[end] < arr[end + 1]:
            while end + 1 < n and arr[end] < arr[end + 1]:
                end += 1
            if end + 1 < n and arr[end] > arr[end + 1]:
                while end + 1 < n and arr[end] > arr[end + 1]:
                    end += 1
                best = max(best, end - base + 1)
        base = max(end, base + 1)
    return best

print(longest_mountain([2, 1, 4, 7, 3, 2, 5]), longest_mountain([2, 2, 2]))   # 5, 0

5 0


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Đoán output
Mảng nào là mảng núi? `[1,2,1]`, `[1,2,3,2,2]`, `[2,3,4,5,4,3,2,1,0]`, `[0,0]`, `[4,3,2,3]`.

### Bài 2 — Kiểm tra tăng nghiêm ngặt bằng vòng lặp (không dùng `all`/`zip`)

In [24]:
def exercise_strictly_increasing(arr: list[int]) -> bool:
    # TODO: duyệt i từ 0 tới len-2, so sánh arr[i] và arr[i+1]
    pass

# assert exercise_strictly_increasing([1, 2, 5]) == True
# assert exercise_strictly_increasing([1, 2, 2]) == False
# assert exercise_strictly_increasing([7]) == True

### Bài 3 — Tự viết lời giải 941 (walk up / walk down)

In [25]:
def exercise_valid_mountain(arr: list[int]) -> bool:
    # TODO 1: leo lên
    # TODO 2: kiểm tra đỉnh không ở đầu/cuối
    # TODO 3: xuống dốc, kiểm tra tới cuối
    pass

# assert exercise_valid_mountain([0, 3, 2, 1]) == True
# assert exercise_valid_mountain([3, 5, 5]) == False
# assert exercise_valid_mountain([0, 1, 2]) == False

### Bài 4 — "Mảng thung lũng": giảm nghiêm ngặt rồi tăng nghiêm ngặt

In [26]:
def exercise_valid_valley(arr: list[int]) -> bool:
    # TODO
    pass

# assert exercise_valid_valley([3, 1, 2]) == True
# assert exercise_valid_valley([1, 2, 3]) == False
# assert exercise_valid_valley([3, 1, 1, 2]) == False

### Bài 5 — Monotonic Array (LeetCode 896) trong một lượt duyệt

In [27]:
def exercise_is_monotonic(arr: list[int]) -> bool:
    # TODO: hai cờ inc, dec; một lượt duyệt
    pass

# assert exercise_is_monotonic([1, 2, 2, 3]) == True
# assert exercise_is_monotonic([1, 3, 2]) == False

### Bài 6 — Mini challenge: số phần tử ít nhất cần xóa để còn lại một mảng núi (LeetCode 1671, bản nhỏ)

Gợi ý tư duy: với mỗi vị trí làm đỉnh, tính dãy con tăng dài nhất **kết thúc** tại đó (từ trái) và dãy con giảm dài nhất **bắt đầu** tại đó (về phải). Với `n <= 1000`, `O(n²)` là đủ.

In [28]:
def exercise_min_removals_mountain(arr: list[int]) -> int:
    # TODO
    pass

# assert exercise_min_removals_mountain([1, 3, 1]) == 0
# assert exercise_min_removals_mountain([2, 1, 1, 5, 6, 2, 3, 1]) == 3

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | walk lên rồi xuống | chú ý đoạn bằng | đỉnh ở đầu/cuối là False |
| 2 | so từng cặp kề | `for i in range(len(arr) - 1)` | `if arr[i] >= arr[i+1]: return False` |
| 3 | 3 bước | `while i + 1 < n and ...` | xem mục 5 |
| 4 | đảo chiều so sánh | xuống trước, lên sau | đổi `<` và `>` |
| 5 | hai cờ | gặp tăng → `dec = False` | `return inc or dec` |
| 6 | LIS hai chiều | `up[i]`, `down[i]` | đỉnh hợp lệ khi `up[i] > 1 and down[i] > 1`; đáp án `n - max(up+down-1)` |

## 17. Đáp án tham khảo

Bài 1: `[1,2,1]` True; `[1,2,3,2,2]` False (bằng ở cuối); `[2,3,4,5,4,3,2,1,0]` True; `[0,0]` False; `[4,3,2,3]` False.

In [29]:
for arr in [[1, 2, 1], [1, 2, 3, 2, 2], [2, 3, 4, 5, 4, 3, 2, 1, 0], [0, 0], [4, 3, 2, 3]]:
    print(arr, mountain_walk(arr))

[1, 2, 1] True
[1, 2, 3, 2, 2] False
[2, 3, 4, 5, 4, 3, 2, 1, 0] True
[0, 0] False
[4, 3, 2, 3] False


In [30]:
def answer_strictly_increasing(arr):
    for i in range(len(arr) - 1):
        if arr[i] >= arr[i + 1]:
            return False
    return True

def answer_valid_mountain(arr):
    n, i = len(arr), 0
    while i + 1 < n and arr[i] < arr[i + 1]:
        i += 1
    if i == 0 or i == n - 1:
        return False
    while i + 1 < n and arr[i] > arr[i + 1]:
        i += 1
    return i == n - 1

def answer_valid_valley(arr):
    n, i = len(arr), 0
    while i + 1 < n and arr[i] > arr[i + 1]:
        i += 1
    if i == 0 or i == n - 1:
        return False
    while i + 1 < n and arr[i] < arr[i + 1]:
        i += 1
    return i == n - 1

def answer_is_monotonic(arr):
    inc = dec = True
    for a, b_ in zip(arr, arr[1:]):
        if a < b_:
            dec = False
        elif a > b_:
            inc = False
    return inc or dec

def answer_min_removals_mountain(arr):
    n = len(arr)
    up = [1] * n
    for i in range(n):
        for j in range(i):
            if arr[j] < arr[i]:
                up[i] = max(up[i], up[j] + 1)
    down = [1] * n
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            if arr[j] < arr[i]:
                down[i] = max(down[i], down[j] + 1)
    best = max(up[i] + down[i] - 1 for i in range(n) if up[i] > 1 and down[i] > 1)
    return n - best

assert answer_strictly_increasing([1, 2, 5]) and not answer_strictly_increasing([1, 2, 2]) and answer_strictly_increasing([7])
assert answer_valid_mountain([0, 3, 2, 1]) and not answer_valid_mountain([3, 5, 5]) and not answer_valid_mountain([0, 1, 2])
assert answer_valid_valley([3, 1, 2]) and not answer_valid_valley([1, 2, 3]) and not answer_valid_valley([3, 1, 1, 2])
assert answer_is_monotonic([1, 2, 2, 3]) and not answer_is_monotonic([1, 3, 2])
assert answer_min_removals_mountain([1, 3, 1]) == 0
assert answer_min_removals_mountain([2, 1, 1, 5, 6, 2, 3, 1]) == 3
for arr, expected in tests:
    assert answer_valid_mountain(arr) == expected
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def validMountainArray(self, arr: list[int]) -> bool:
        n, i = len(arr), 0
        while i + 1 < n and arr[i] < arr[i + 1]:   # leo lên
            i += 1
        if i == 0 or i == n - 1:                     # đỉnh ở chân núi
            return False
        while i + 1 < n and arr[i] > arr[i + 1]:   # xuống dốc
            i += 1
        return i == n - 1                            # tới chân phải?
```

**Câu thần chú:** *"Leo hết dốc lên, đỉnh không ở chân, xuống hết dốc phải chạm chân bên kia."*

**Checklist trước khi submit:**
- [ ] So sánh **nghiêm ngặt** `<`, `>`.
- [ ] Kiểm tra đỉnh `i != 0` và `i != n - 1`.
- [ ] Có `i + 1 < n` trước khi đọc `arr[i + 1]`.
- [ ] Mảng `n < 3` tự động `False` (hoặc kiểm tra riêng).
- [ ] Test: chỉ tăng, chỉ giảm, đoạn bằng, núi hai đỉnh.